# Experiment 01 — PubMed candidate abstract retrieval

**Goal:** define and sanity-check a query + fetch strategy that retrieves candidate
abstracts about **dietary supplements and health claims**, for downstream PICO
extraction and ontology matching.

Retrieval is anchored to **claims** (substance x health outcome) rather than to
supplements alone. Every abstract therefore carries the `claim_id` that surfaced it,
so a later PICO extraction can be checked against the claim we actually searched for.

**What to look for while running:**
1. Are the returned abstracts on-topic? (precision, eyeball)
2. How much does the study-type filter cost? (recall, measured in section 4)
3. Does PubMed's query *translation* match what we intended? (section 3)

In [1]:
%load_ext autoreload
%autoreload 2

import os, sys, json, textwrap
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

from engine.services.sources.pubmed import (
    QuerySpec, PubMedClient, run_query, save_run,
    new_run_dir, save_run_manifest,
    INTERVENTIONAL_PT, EXCLUDED_PT,
)

client = PubMedClient()
print("API key loaded:", bool(client.api_key), "| rate limit:", round(1/client._min_interval, 1), "req/s")

API key loaded: True | rate limit: 6.7 req/s


## 1. Experiment parameters

Everything tunable lives here. Re-run from this cell after changing anything.

In [2]:
# --- fetch controls -------------------------------------------------------
LIMIT_PER_CLAIM = 25     # max abstracts downloaded per claim (None = all hits)
MAX_CLAIMS      = 8      # how many claims to run in this experiment
BATCH_SIZE      = 200    # efetch page size

# --- filter controls ------------------------------------------------------
INTERVENTIONAL_ONLY  = True   # restrict to trial / meta-analysis / systematic review
HUMANS_ONLY          = True
EXCLUDE_NON_RESEARCH = True   # drop editorials, comments, letters, retractions
REQUIRE_ABSTRACT     = True
ENGLISH_ONLY         = True
DATE_FROM            = None   # e.g. "2000/01/01"
DATE_TO              = None

# --- output ---------------------------------------------------------------
# Each execution gets its own timestamped directory, so runs never overwrite
# one another and two runs can be diffed directly.
RUNS_DIR  = ROOT / "data" / "pubmed" / "runs"
RUN_LABEL = "seed8-strict"   # short tag appended to the timestamp

FILTERS = dict(
    interventional_only=INTERVENTIONAL_ONLY,
    humans_only=HUMANS_ONLY,
    exclude_non_research=EXCLUDE_NON_RESEARCH,
    require_abstract=REQUIRE_ABSTRACT,
    english_only=ENGLISH_ONLY,
    date_from=DATE_FROM,
    date_to=DATE_TO,
)
print(json.dumps(FILTERS, indent=2))

{
  "interventional_only": true,
  "humans_only": true,
  "exclude_non_research": true,
  "require_abstract": true,
  "english_only": true,
  "date_from": null,
  "date_to": null
}


## 2. Seed claims

Hand-written seed set for this first experiment: well-known supplement/outcome pairs
spanning botanicals, vitamins, minerals and fatty acids.

**Why hand-written and not driven off the EU register yet:** the register's
`health_relation` column is null on 433 of 3,342 rows, and its `claim` text is
regulatory prose ("contributes to the normal function of the immune system") rather
than search vocabulary. Mapping that prose to MeSH is its own task. This seed set
isolates the *retrieval* question from the *claim-normalisation* question.

In [3]:
SEED_CLAIMS = [
    dict(claim_id="SEED-001", substance="Serenoa repens",
         substance_synonyms=["saw palmetto"],
         health_terms=["benign prostatic hyperplasia", "lower urinary tract symptoms"],
         claim_text="Saw palmetto for benign prostatic hyperplasia"),
    dict(claim_id="SEED-002", substance="Melatonin",
         substance_synonyms=[],
         health_terms=["sleep quality", "insomnia", "sleep latency"],
         claim_text="Melatonin contributes to the reduction of time taken to fall asleep"),
    dict(claim_id="SEED-003", substance="Docosahexaenoic acid",
         substance_synonyms=["DHA", "omega-3 fatty acids"],
         health_terms=["brain development", "cognitive function"],
         claim_text="DHA contributes to normal brain development"),
    dict(claim_id="SEED-004", substance="Vitamin D",
         substance_synonyms=["cholecalciferol", "ergocalciferol"],
         health_terms=["bone mineral density", "osteoporosis", "fracture"],
         claim_text="Vitamin D helps maintain normal bones"),
    dict(claim_id="SEED-005", substance="Curcuma longa",
         substance_synonyms=["curcumin", "turmeric"],
         health_terms=["inflammation", "osteoarthritis", "joint pain"],
         claim_text="Curcumin for joint health and inflammation"),
    dict(claim_id="SEED-006", substance="Probiotics",
         substance_synonyms=["Lactobacillus", "Bifidobacterium"],
         health_terms=["irritable bowel syndrome", "gut microbiota"],
         claim_text="Probiotics support digestive health"),
    dict(claim_id="SEED-007", substance="Zinc",
         substance_synonyms=["zinc supplementation"],
         health_terms=["common cold", "immune function"],
         claim_text="Zinc contributes to the normal function of the immune system"),
    dict(claim_id="SEED-008", substance="Ginkgo biloba",
         substance_synonyms=["ginkgo"],
         health_terms=["cognitive function", "memory", "dementia"],
         claim_text="Ginkgo biloba for cognitive function"),
]

specs = [QuerySpec(**c, **FILTERS) for c in SEED_CLAIMS[:MAX_CLAIMS]]
print(f"{len(specs)} claims queued")

8 claims queued


## 3. Inspect queries before spending requests

`querytranslation` is PubMed's own expansion of the query — it reveals which MeSH
terms were silently added. Worth reading: it is often **not** what you intended,
and that is the cheapest place to catch a bad query.

In [4]:
rows = []
for s in specs:
    res = client.search(s.build())
    rows.append(dict(claim_id=s.claim_id, substance=s.substance, hits=res["count"]))
    print(f"[{s.claim_id}] {s.substance} -> {res['count']} hits")

preview = pd.DataFrame(rows)
print()
print(f"total hits across claims: {preview.hits.sum():,}")
preview

[SEED-001] Serenoa repens -> 73 hits
[SEED-002] Melatonin -> 542 hits
[SEED-003] Docosahexaenoic acid -> 142 hits
[SEED-004] Vitamin D -> 2036 hits
[SEED-005] Curcuma longa -> 310 hits
[SEED-006] Probiotics -> 1621 hits
[SEED-007] Zinc -> 101 hits
[SEED-008] Ginkgo biloba -> 190 hits

total hits across claims: 5,015


,claim_id,substance,hits
0,SEED-001,Serenoa repens,73
1,SEED-002,Melatonin,542
2,SEED-003,Docosahexaenoic acid,142
3,SEED-004,Vitamin D,2036
4,SEED-005,Curcuma longa,310
5,SEED-006,Probiotics,1621
6,SEED-007,Zinc,101
7,SEED-008,Ginkgo biloba,190


In [5]:
# Full query + PubMed's translation for one claim, to audit term expansion.
s = specs[0]
print("QUERY AS BUILT:\n")
print(textwrap.fill(s.build(), 100))
print("\n" + "="*100)
print("\nPUBMED TRANSLATION:\n")
print(textwrap.fill(client.search(s.build())["translation"] or "(none)", 100))

QUERY AS BUILT:

("Serenoa repens"[tiab] OR "Serenoa repens"[mh] OR "saw palmetto"[tiab] OR "saw palmetto"[mh]) AND
("benign prostatic hyperplasia"[tiab] OR "benign prostatic hyperplasia"[mh] OR "lower urinary tract
symptoms"[tiab] OR "lower urinary tract symptoms"[mh]) AND (Randomized Controlled Trial[pt] OR
Controlled Clinical Trial[pt] OR Clinical Trial[pt] OR Meta-Analysis[pt] OR Systematic Review[pt])
AND Humans[mh] AND hasabstract AND English[lang] NOT Editorial[pt] NOT Comment[pt] NOT Letter[pt]
NOT News[pt] NOT Retracted Publication[pt]


PUBMED TRANSLATION:

(((((("Serenoa repens"[Title/Abstract] OR "saw palmetto"[Title/Abstract]) AND ("benign prostatic
hyperplasia"[Title/Abstract] OR "lower urinary tract symptoms"[Title/Abstract] OR "lower urinary
tract symptoms"[MeSH Terms]) AND ("randomized controlled trial"[Publication Type] OR "controlled
clinical trial"[Publication Type] OR "clinical trial"[Publication Type] OR "Meta-
Analysis"[Publication Type] OR "systematic review"[Pu

## 4. What does the study-type filter cost?

The publication-type filter is the biggest single lever in the strategy. It raises
precision (PICO on an editorial or a mouse study yields garbage) but silently
discards real evidence.

This section **measures** the cost instead of assuming it. A high discard rate is not
automatically bad — but it should be a decision, not an accident.

In [6]:
cost_rows = []
for c in SEED_CLAIMS[:MAX_CLAIMS]:
    strict  = QuerySpec(**c, **{**FILTERS, "interventional_only": True})
    relaxed = QuerySpec(**c, **{**FILTERS, "interventional_only": False})
    n_strict  = client.search(strict.build())["count"]
    n_relaxed = client.search(relaxed.build())["count"]
    cost_rows.append(dict(
        claim_id=c["claim_id"], substance=c["substance"],
        strict=n_strict, relaxed=n_relaxed,
        discarded=n_relaxed - n_strict,
        kept_pct=round(100 * n_strict / n_relaxed, 1) if n_relaxed else None,
    ))

cost = pd.DataFrame(cost_rows)
print(f"overall kept: {100*cost.strict.sum()/cost.relaxed.sum():.1f}% "
      f"({cost.strict.sum():,} of {cost.relaxed.sum():,})")
cost

overall kept: 18.7% (5,015 of 26,811)


,claim_id,substance,strict,relaxed,discarded,kept_pct
0,SEED-001,Serenoa repens,73,199,126,36.7
1,SEED-002,Melatonin,542,1655,1113,32.7
2,SEED-003,Docosahexaenoic acid,142,546,404,26.0
3,SEED-004,Vitamin D,2036,11336,9300,18.0
4,SEED-005,Curcuma longa,310,2260,1950,13.7
5,SEED-006,Probiotics,1621,9831,8210,16.5
6,SEED-007,Zinc,101,470,369,21.5
7,SEED-008,Ginkgo biloba,190,514,324,37.0


## 5. Fetch abstracts

`LIMIT_PER_CLAIM` caps downloads. The manifest still records the **full** hit count,
so a truncated run reports what it truncated rather than looking complete.

In [7]:
RUN_DIR = new_run_dir(RUNS_DIR, RUN_LABEL)
print(f"run: {RUN_DIR.name}\n")

all_abstracts, manifests = [], []

for s in specs:
    abstracts, manifest = run_query(s, client=client, limit=LIMIT_PER_CLAIM,
                                    batch_size=BATCH_SIZE)
    all_abstracts.extend(abstracts)
    manifests.append(manifest)
    paths = save_run(abstracts, manifest, RUN_DIR, run_name=s.claim_id)
    print(f"[{s.claim_id}] fetched {manifest['fetched']:>3} of {manifest['total_hits']:>5} hits"
          f"  -> {paths['raw'].name}")

run_manifest = save_run_manifest(
    manifests, RUN_DIR,
    params={**FILTERS, "limit_per_claim": LIMIT_PER_CLAIM,
            "max_claims": MAX_CLAIMS, "label": RUN_LABEL},
)
print(f"\ntotal abstracts fetched: {len(all_abstracts)}")
print(f"run manifest: {run_manifest}")

run: 20260902T161349Z-seed8-strict



[SEED-001] fetched  25 of    73 hits  -> SEED-001.jsonl


[SEED-002] fetched  25 of   542 hits  -> SEED-002.jsonl


[SEED-003] fetched  25 of   142 hits  -> SEED-003.jsonl


[SEED-004] fetched  25 of  2036 hits  -> SEED-004.jsonl


[SEED-005] fetched  25 of   310 hits  -> SEED-005.jsonl


[SEED-006] fetched  25 of  1621 hits  -> SEED-006.jsonl


[SEED-007] fetched  25 of   101 hits  -> SEED-007.jsonl


[SEED-008] fetched  25 of   190 hits  -> SEED-008.jsonl

total abstracts fetched: 200
run manifest: /var/tmp/ge74wan/repos/supplement-kg/data/pubmed/runs/20260902T161349Z-seed8-strict/run.json


## 6. Inspect what came back

The eyeball test. Read a few and ask: *could a PICO frame be extracted from this,
and does it concern the claim we searched for?*

In [8]:
df = pd.DataFrame([{
    "pmid": a.pmid, "claim_id": a.claim_id, "year": a.year,
    "title": a.title, "journal": a.journal,
    "pub_types": "; ".join(a.publication_types),
    "n_mesh": len(a.mesh_terms),
    "abstract_chars": len(a.abstract),
    "structured": a.abstract.split(":")[0].isupper() if ":" in a.abstract else False,
} for a in all_abstracts])

print("rows:", len(df))
print("unique PMIDs:", df.pmid.nunique(), "| duplicates across claims:", len(df) - df.pmid.nunique())
print("empty abstracts:", (df.abstract_chars == 0).sum())
print("structured abstracts:", f"{100*df.structured.mean():.0f}%")
df.head(15)

rows: 200
unique PMIDs: 199 | duplicates across claims: 1
empty abstracts: 0
structured abstracts: 64%


,pmid,claim_id,year,title,journal,pub_types,n_mesh,abstract_chars,structured
0,42280395,SEED-001,2026,Efficacy and Safety of Salvia miltiorrhiza Ext...,Nutrients,Journal Article; Randomized Controlled Trial; ...,15,1610,False
1,41098072,SEED-001,2026,Efficacy of Serenoa repens Extract Combined Wi...,The Prostate,Journal Article; Multicenter Study; Randomized...,15,2352,True
2,39675037,SEED-001,2024,Measuring the efficacy of Serenoa repens (USPl...,The Canadian journal of urology,Journal Article; Clinical Trial,12,1321,True
3,39042923,SEED-001,2025,Beneficial effects of saw palmetto (Serenoa re...,Nutrition and health,Journal Article; Randomized Controlled Trial,16,1579,False
4,35334848,SEED-001,2022,Beneficial Effects of Saw Palmetto Fruit Extra...,Nutrients,Journal Article; Multicenter Study; Randomized...,9,1373,False
5,34933534,SEED-001,2021,Serenoa repens and its effects on male sexual ...,"Archivio italiano di urologia, andrologia : or...",Journal Article; Meta-Analysis; Systematic Review,9,1788,True
6,34488251,SEED-001,2021,Serenoa repens for the treatment of lower urin...,Investigative and clinical urology,Journal Article; Meta-Analysis; Systematic Review,6,1347,True
7,32274957,SEED-001,2020,Comparison of Serenoa repens With Tamsulosin i...,American journal of men's health,Journal Article; Meta-Analysis; Research Suppo...,11,1681,False
8,31952967,SEED-001,2021,Clinical Efficacy of Serenoa repens Versus Pla...,European urology focus,Journal Article; Network Meta-Analysis; System...,10,2277,True
9,31451276,SEED-001,2019,Benign prostate hyperplasia and nutrition.,Clinical nutrition ESPEN,Journal Article; Systematic Review,14,1083,True


In [9]:
# Publication-type mix -- confirms the filter did what we asked.
from collections import Counter
pt = Counter(p for a in all_abstracts for p in a.publication_types)
pd.Series(dict(pt.most_common(15)), name="count").to_frame()

,count
Journal Article,200
Randomized Controlled Trial,99
Systematic Review,89
Meta-Analysis,42
"Research Support, Non-U.S. Gov't",27
Multicenter Study,16
Comparative Study,12
Network Meta-Analysis,8
Clinical Trial,5
"Clinical Trial, Phase IV",4


In [10]:
# Read a few in full.
for a in all_abstracts[:3]:
    print("="*100)
    print(f"PMID {a.pmid}  ({a.year})  [{a.claim_id}]  {'; '.join(a.publication_types[:3])}")
    print(f"TITLE: {a.title}")
    print(f"\n{textwrap.fill(a.abstract[:1200], 100)}")
    print(f"\nMeSH: {', '.join(a.mesh_terms[:12])}")
    print()

PMID 42280395  (2026)  [SEED-001]  Journal Article; Randomized Controlled Trial; Comparative Study
TITLE: Efficacy and Safety of Salvia miltiorrhiza Extract (SAGX) Compared with Saw Palmetto in Men with Lower Urinary Tract Symptoms: A 12-Week, Randomized, Double-Blind, Parallel-Group Pilot Study.

Background: Lower urinary tract symptoms (LUTS) are prevalent among aging men and negatively affect
quality of life. Salvia miltiorrhiza extract (SAGX), which exhibits anti-inflammatory and
antioxidant properties, has been developed as a functional ingredient to improve LUTS. However,
comparative clinical data against established standards, such as saw palmetto, are currently
lacking. Objectives: To compare the efficacy and safety of SAGX with saw palmetto extract in men
with LUTS. Methods: In this randomized, parallel-group pilot study, 30 men with LUTS were assigned
to receive either 400 mg of SAGX or 320 mg of saw palmetto extract once daily for 12 weeks. The
primary outcome was the change

## 7. Overlap between claims

The same PMID can surface under several claims. That is legitimate provenance
(one paper can support several claims), but it decides whether the corpus is stored
per-claim or as a deduplicated pool with a claim->PMID mapping alongside.

In [11]:
dupes = (df.groupby("pmid")["claim_id"].agg(["nunique", lambda s: sorted(set(s))])
           .rename(columns={"nunique": "n_claims", "<lambda_0>": "claims"}))
dupes = dupes[dupes.n_claims > 1].sort_values("n_claims", ascending=False)
print(f"{len(dupes)} PMIDs appear under more than one claim")
dupes.head(10)

1 PMIDs appear under more than one claim


,n_claims,claims
pmid,,
39317299,2,"[SEED-003, SEED-008]"


## 8. Run summary

Manifests are the committable artifact: query, translation, hit count, timestamp,
sorted PMID list and hash. Raw abstracts stay gitignored under `data/`.

In [12]:
summary = pd.DataFrame([{
    "claim_id": m["claim_id"], "substance": m["substance"],
    "total_hits": m["total_hits"], "fetched": m["fetched"],
    "pmid_hash": m["pmid_hash"],
} for m in manifests])

run_doc = json.loads((RUN_DIR / "run.json").read_text())
print(f"run id:       {run_doc['run_id']}")
print(f"corpus hash:  {run_doc['corpus_hash']}")
print(f"unique PMIDs: {run_doc['unique_pmids']}")
print(f"location:     {RUN_DIR}")
print(f"\nfetched {summary.fetched.sum()} abstracts from {summary.total_hits.sum():,} total hits")
summary

run id:       20260902T161349Z-seed8-strict
corpus hash:  97e02863a05f2d40
unique PMIDs: 199
location:     /var/tmp/ge74wan/repos/supplement-kg/data/pubmed/runs/20260902T161349Z-seed8-strict

fetched 200 abstracts from 5,015 total hits


,claim_id,substance,total_hits,fetched,pmid_hash
0,SEED-001,Serenoa repens,73,25,e8fd215dade4937d
1,SEED-002,Melatonin,542,25,e936e0f837c7c594
2,SEED-003,Docosahexaenoic acid,142,25,d868aba32a62b6c3
3,SEED-004,Vitamin D,2036,25,eef5d7e33a3ffc9e
4,SEED-005,Curcuma longa,310,25,b1d0fb0261e26d35
5,SEED-006,Probiotics,1621,25,11475c89d0433b53
6,SEED-007,Zinc,101,25,1ddc9d39a0191ee4
7,SEED-008,Ginkgo biloba,190,25,36b764b9ba0de097


### Comparing runs

Every execution leaves an immutable timestamped directory. Change a filter,
re-run, and compare `corpus_hash` to see whether the change actually moved the
retrieved set.

In [13]:
# All runs to date. `corpus_hash` changes iff the retrieved PMID set changed,
# so two rows with different params and the same hash mean the change was inert.
run_rows = []
for rj in sorted(RUNS_DIR.glob("*/run.json")):
    d = json.loads(rj.read_text())
    run_rows.append({
        "run_id": d["run_id"],
        "queries": d["n_queries"],
        "hits": d["total_hits"],
        "fetched": d["total_fetched"],
        "unique_pmids": d["unique_pmids"],
        "corpus_hash": d["corpus_hash"],
        "interventional": d["params"].get("interventional_only"),
        "limit": d["params"].get("limit_per_claim"),
    })
pd.DataFrame(run_rows)

,run_id,queries,hits,fetched,unique_pmids,corpus_hash,interventional,limit
0,20260902T161349Z-seed8-strict,8,5015,200,199,97e02863a05f2d40,True,25


## Open questions this run should inform

1. **Multi-substance claims** — `_or_block` ORs the substance with its synonyms, so
   *"Calcium and vitamin D"* retrieves calcium **or** vitamin D, not the combination.
   Many EU register claims are of this form. Needs an AND-group representation.
2. **Claim text -> search vocabulary** — regulatory prose is not query vocabulary.
   Needs a mapping step to MeSH before the register can drive retrieval directly.
3. **Filter cost** — see section 4. Is the discard rate acceptable, or should
   observational studies be admitted with a study-design tag?
4. **Corpus shape** — per-claim files, or one deduplicated pool plus a claim mapping?